In [ ]:
!pip install -U transformers

# 1. Run the model Gemma-2-2b-it from huggingface


## Local Inference on GPU
Model page: https://huggingface.co/google/gemma-2-2b-it

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/google/gemma-2-2b-it)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
from huggingface_hub import login
!pip install -U bitsandbytes>=0.46.1
login()

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata
# 1. Install libraries required for 4-bit loading
!pip install -q -U accelerate transformers


# Moved model and tokenizer loading into this cell to ensure they are always available.
print("Loading Gemma model and tokenizer...")
hf_token = userdata.get('HF_TOKEN')
model_id = "google/gemma-2-2b-it"


tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    token=hf_token,
    output_hidden_states=True
)
print("Gemma model and tokenizer loaded.")

Loading Gemma model and tokenizer...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Gemma model and tokenizer loaded.


In [ ]:
# Format your input message using Gemma's chat template
messages = [
    {"role": "user", "content": "Hello! Can you tell me what toxicity means in online communities?"}
]

# Apply the chat template to format the prompt correctly
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize and move to the model's device
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate a response
outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7
)

# Decode and print only the newly generated response (skipping the input prompt)
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("Gemma Response:\n", response)

Gemma Response:
 "Toxicity" in online communities refers to **negative, harmful, and disruptive behavior** that can create a hostile, unpleasant, or even dangerous environment for other members. 

Here's a breakdown of what constitutes toxicity:

**Types of Toxicity:**

* **Harassment:**  This includes verbal or visual abuse, threats, stalking, doxing, hate speech, and other forms of intimidation or bullying.
* **Cyberbullying:**  This is a more specific form of harassment, often involving repeated attacks and negativity directed at someone online.
* **Discrimination:**  Toxicity can take the form of prejudice and exclusion based on factors like race, gender, sexual orientation, religion, disability, etc.
* **Spam & Disinformation


#2. Load the dataset = "thesofakillers/jigsaw-toxic-comment-classification-challenge"

In [ ]:
from datasets import load_dataset

import pandas as pd

# Use the direct resolve link that forces Git LFS to download the actual CSV payload
url = "https://huggingface.co/datasets/thesofakillers/jigsaw-toxic-comment-classification-challenge/resolve/main/train.csv"

print("Downloading actual dataset...")
df = pd.read_csv(url).head(200)

print("Columns found now:", df.columns.tolist())
print(df.head(2))

Columns found now: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  


#3. Extract the layers

In [ ]:
import torch
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

# 1. Extract Gemma hidden state embeddings
def extract_gemma_hidden_states(dataframe, model, tokenizer, max_length=512):
    features = []
    labels = []

    print(f"Starting extraction for {len(dataframe)} samples...")
    for idx, row in dataframe.iterrows():
        text = str(row['comment_text'])
        label = int(row['toxic'])

        if not text.strip():
            text = "empty"

        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        # Extract final hidden state of the last token from the last layer
        last_hidden_state = outputs.hidden_states[-1]
        vec = last_hidden_state[:, -1, :].to(torch.float32).cpu().numpy()

        features.append(vec)
        labels.append(label)

    return np.vstack(features), np.array(labels)

X_train, y_train = extract_gemma_hidden_states(df, model, tokenizer)
print("✅ Feature extraction complete! X_train shape:", X_train.shape)

Starting extraction for 200 samples...
✅ Feature extraction complete! X_train shape: (200, 2304)


#4. Train  a minimal model classifer.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import numpy as np
import torch
import joblib
from datasets import load_dataset, Dataset, Features, ClassLabel
from sklearn.model_selection import train_test_split # Import train_test_split from sklearn

# 1. Define the feature extraction function using the global model and tokenizer
def extract_layer_features(text, model, tokenizer, max_length=512):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    # Extract final hidden state of the last token from the last layer
    last_hidden_state = outputs.hidden_states[-1]
    return last_hidden_state[:, -1, :].to(torch.float32).cpu().numpy()

def get_features_labels_from_dataset_split(dataset_split: Dataset, text_col='comment_text', label_col='toxic', max_samples=None):
    features = []
    labels = []

    if max_samples is not None:
        dataset_subset = dataset_split.select(range(min(max_samples, len(dataset_split))))
    else:
        dataset_subset = dataset_split

    print(f"Extracting features from {len(dataset_subset)} samples...")
    for i, item in enumerate(dataset_subset):
        text = str(item[text_col])
        label = int(item[label_col]) # Label is already binarized and cast to ClassLabel

        if not text.strip():
            text = "empty"

        vector = extract_layer_features(text, model, tokenizer)
        features.append(vector)
        labels.append(label)
        if (i + 1) % 100 == 0:
            print(f"  Processed {i + 1}/{len(dataset_subset)} samples.")

    return np.vstack(features), np.array(labels)

print("Loading dataset for training, validation, and testing splits...")
# Load the full dataset into a pandas DataFrame directly from the URL
url = "https://huggingface.co/datasets/thesofakillers/jigsaw-toxic-comment-classification-challenge/resolve/main/train.csv"
df_full = pd.read_csv(url)

# Binarize labels
df_full['toxic'] = df_full['toxic'].apply(lambda x: 1 if float(x) > 0.5 else 0)

# Create stratified train/validation/test splits using sklearn
# First, split into 80% train and 20% temp (for val/test)
train_df, temp_df = train_test_split(
    df_full,
    test_size=0.2,
    stratify=df_full['toxic'], # Stratify by 'toxic' label for better balance
    random_state=42
)

# Then, split the temp_df into 50% validation and 50% test (10% each of original)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5, # 0.5 of the 20% is 10%
    stratify=temp_df['toxic'], # Stratify again
    random_state=42
)

# Convert pandas DataFrames back to datasets.Dataset objects
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Define features and cast 'toxic' column to ClassLabel for consistency if needed for other dataset operations
# Although sklearn split already handled the stratification, this ensures the 'toxic' column in the Dataset objects
# has the correct type if future dataset operations require it.
features_with_class_label = train_dataset.features.copy()
features_with_class_label['toxic'] = ClassLabel(num_classes=2, names=['nontoxic', 'toxic'])

train_dataset = train_dataset.cast(features_with_class_label)
val_dataset = val_dataset.cast(features_with_class_label)
test_dataset = test_dataset.cast(features_with_class_label)

print(f"Dataset split: Train {len(train_dataset)} samples, Validation {len(val_dataset)} samples, Test {len(test_dataset)} samples")

# 2. Extract features across splits
print("\nExtracting training features...")
X_train, y_train = get_features_labels_from_dataset_split(train_dataset, max_samples=2000)

print("\nExtracting validation features...")
X_val, y_val = get_features_labels_from_dataset_split(val_dataset, max_samples=400)

print("\nExtracting testing features...")
X_test, y_test = get_features_labels_from_dataset_split(test_dataset, max_samples=400)

print("\nFeature extraction complete for all splits!")
print("Training feature matrix shape:", X_train.shape)
print("Training labels shape:", y_train.shape)
print("Validation feature matrix shape:", X_val.shape)
print("Validation labels shape:", y_val.shape)
print("Testing feature matrix shape:", X_test.shape)
print("Testing labels shape:", y_test.shape)

# 3. Initialize and train the classifier
classifier = LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')
print("\nTraining classifier...")
classifier.fit(X_train, y_train)
print("Classifier trained successfully!")

# 4. Evaluate on Validation Set
print("\nEvaluating on Validation Set:")
y_val_pred = classifier.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(y_val, y_val_pred) * 100:.2f}%")
print("Validation Classification Report:\n", classification_report(y_val, y_val_pred))

# 5. Evaluate on Test Set
print("\nEvaluating on Test Set:")
y_test_pred = classifier.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, y_test_pred) * 100:.2f}%")
print("Test Classification Report:\n", classification_report(y_test, y_test_pred))

# 6. Save the trained probe as joblib and write classifier.py
joblib.dump(classifier, 'trained_probe.joblib')
print("\n✅ trained_probe.joblib saved successfully!")

classifier_code = '''import numpy as np
import joblib
import os


class Classifier:
    """
    Trained toxicity linear probe classifier.
    Loads the saved joblib model from disk and uses it to predict
    toxicity labels on extracted feature vectors.
    """

    def __init__(self):
        model_path = os.path.join(os.path.dirname(__file__), 'trained_probe.joblib')
        self.model = joblib.load(model_path)

    def predict(self, X):
        X = np.asarray(X)
        return self.model.predict(X)
'''

with open('classifier.py', 'w') as f:
    f.write(classifier_code)

print("✅ classifier.py generated successfully!")

Loading dataset for training, validation, and testing splits...


Casting the dataset:   0%|          | 0/127656 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/15957 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/15958 [00:00<?, ? examples/s]

Dataset split: Train 127656 samples, Validation 15957 samples, Test 15958 samples

Extracting training features...
Extracting features from 2000 samples...
  Processed 100/2000 samples.
  Processed 200/2000 samples.
  Processed 300/2000 samples.
  Processed 400/2000 samples.
  Processed 500/2000 samples.
  Processed 600/2000 samples.
  Processed 700/2000 samples.
  Processed 800/2000 samples.
  Processed 900/2000 samples.
  Processed 1000/2000 samples.
  Processed 1100/2000 samples.
  Processed 1200/2000 samples.
  Processed 1300/2000 samples.
  Processed 1400/2000 samples.
  Processed 1500/2000 samples.
  Processed 1600/2000 samples.
  Processed 1700/2000 samples.
  Processed 1800/2000 samples.
  Processed 1900/2000 samples.
  Processed 2000/2000 samples.

Extracting validation features...
Extracting features from 400 samples...
  Processed 100/400 samples.
  Processed 200/400 samples.
  Processed 300/400 samples.
  Processed 400/400 samples.

Extracting testing features...
Extracting

# 5. Test the classifer


In [ ]:
# 1. Define your new sentence
new_text = "You do deserve it"

# 2. Extract features using Gemma
inputs = tokenizer(new_text, return_tensors="pt", truncation=True, max_length=512).to(model.device)
with torch.no_grad():
    outputs = model(**inputs)
vector = outputs.hidden_states[-1][:, -1, :].to(torch.float32).cpu().numpy()

# 3. Check for toxicity with your trained classifier
prediction = classifier.predict(vector)[0]
confidence = classifier.predict_proba(vector)[0][prediction] * 100

# 4. Display the result
status = "Toxic ⚠️" if prediction == 1 else "Safe ✅"
print(f"Prediction: {status} ({confidence:.2f}% confidence)")

Prediction: Safe ✅ (92.11% confidence)


In [ ]:
import zipfile

# Create a clean zip archive with files at the absolute root level
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('classifier.py', arcname='classifier.py')
    zipf.write('trained_probe.joblib', arcname='trained_probe.joblib')

print("✅ submission.zip created successfully with flat root structure!")

✅ submission.zip created successfully with flat root structure!
